##### ktor-client, dataframe, kandy, org.json.XML, org.sqlite.JDBC
* Convert XML data collected via the ktor client to JSON type and load it using DataFrame.readJson.
* Load the SQLite table using DataFrame.readSqlQuery.
* Then, join the two dataframes and create a chart using Kandy.

In [19]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [20]:
@file:DependsOn("org.json:json:20250107")
@file:DependsOn("org.xerial:sqlite-jdbc:3.49.1.0")
@file:DependsOn("ch.qos.logback:logback-classic:1.5.12")

In [21]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets
import java.sql.Connection
import java.sql.DriverManager
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter
import org.json.XML

In [22]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [23]:
Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection(configData.SQLITE_DB.jdbcURL)

In [24]:
val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(12)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-26 23:16:28, wtch_dt_end:2026-07-27 11:16:28


In [25]:
val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [26]:
fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}

In [27]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Number,2080,1101,0,4.090000,11,6.367079,1.841130,3.210000,4.770000,6.564000,7.650000,11.910000
rtmWqChpla,Comparable<*>,2080,1393,0,,138,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2080,1,0,,2080,null,null,,,,,
rtmWqWtchStaCd,String,2080,15,0,NEP2002,141,null,null,NEP1002,NEP3001,SEA1301,SEA5002,SEA7002
num,Int,2080,2080,0,1,1,1040.500000,600.588600,1,520.416667,1040.500000,1560.583333,2080
rtmWqTu,Int,2080,127,0,1,281,108.622596,361.247729,-1,3.000000,6.000000,18.000000,1477
ph,Number,2080,184,0,7.620000,51,7.863048,0.318202,5.880000,7.620000,7.800000,8.130000,8.610000
rtmWqSlnty,Number,2080,1908,0,31.014000,9,25.209035,7.140906,0.090000,20.674000,27.568001,31.014000,33.470001
rtmWqCndctv,Number,2080,1948,0,47.830002,5,39.695987,10.505825,0.189000,34.195999,43.410000,47.743000,53.147999
rtmWqWtchDtlDt,String,2080,141,0,2026-07-26 23:20:00.0,15,null,null,2026-07-26 23:20:00.0,2026-07-27 02:10:00.0,2026-07-27 05:05:00.0,2026-07-27 07:55:00.0,2026-07-27 11:00:00.0


In [28]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert {
    rtmWqDoxn and rtmWqChpla and rtmWqSlnty and rtmWqCndctv and rtmWtchWtem and ph
}.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else  value.toDouble()
}.convert {
    rtmWqTu
}.with{
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0 else value.toInt()
}

df.schema()

rtmWqDoxn: Double
rtmWqChpla: Double
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: Double
rtmWqSlnty: Double
rtmWqCndctv: Double
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: Double

In [29]:
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2080,1100,0,4.090000,11,6.367079,1.841130,3.210000,4.770000,6.567000,7.650000,11.910000
rtmWqChpla,Double,2080,1393,0,0.000000,138,9.263371,14.137646,0.000000,2.050000,5.425000,9.225833,71.040000
rtmWqWtchStaCd,String,2080,15,0,NEP2002,141,null,null,NEP1002,NEP3001,SEA1301,SEA5002,SEA7002
num,Int,2080,2080,0,1,1,1040.500000,600.588600,1,520.416667,1040.500000,1560.583333,2080
rtmWqTu,Int,2080,127,0,1,281,108.622596,361.247729,-1,3.000000,6.000000,18.000000,1477
ph,Double,2080,182,0,7.620000,51,7.863048,0.318202,5.880000,7.620000,7.800000,8.130000,8.610000
rtmWqSlnty,Double,2080,1907,0,31.014000,9,25.209035,7.140906,0.090000,20.674416,27.569000,31.014000,33.470001
rtmWqCndctv,Double,2080,1948,0,47.830000,5,39.695987,10.505825,0.189000,34.202250,43.411499,47.747666,53.148000
rtmWqWtchDtlDt,LocalDateTime,2080,141,0,2026-07-26T23:20,15,null,null,2026-07-26T23:20,2026-07-27T02:10,2026-07-27T05:05,2026-07-27T07:55,2026-07-27T11:00
rtmWtchWtem,Double,2080,677,0,29.559999,21,28.054788,2.100361,21.990000,26.730000,28.559999,29.559999,32.490002


In [30]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [31]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-26T23:20,7.080000,1.650000,NEP2002,7,7.670000,30.736000,48.004002,25.790001
2,2026-07-26T23:20,10.220000,15.640000,SEA1006,8,7.970000,22.374000,35.653000,29.180000
3,2026-07-26T23:20,6.390000,56.710000,SEA1301,4,8.020000,31.106000,46.547001,29.230000
4,2026-07-26T23:20,7.220000,7.640000,SEA7002,93,7.630000,26.429000,39.646999,22.990000
5,2026-07-26T23:20,3.960000,2.620000,SEA5003,4,7.740000,32.436000,50.937000,26.370001


In [32]:
Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite")

In [33]:
val sqlStmt = "SELECT * FROM OWQObservatory"
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sta_code,String,19,19,0,SEA1002,1,null,null,NEP1001,NEP3001,SEA1301,SEA3003,SEA7002
sta_name,String,19,19,0,시화조력,1,null,null,광양망덕,낙동명지,부산수영,영산목포,천수만
ocean_division,String,19,2,0,특별관리해역,12,null,null,특별관리해역,특별관리해역,특별관리해역,하구 및 만,하구 및 만
lon,Double,19,19,0,126.611000,1,127.588263,1.113453,126.366000,126.540167,127.605000,128.615167,129.387000
lat,Double,19,18,0,35.802000,2,35.687263,0.981615,34.782000,34.990667,35.211000,35.947833,37.731000


In [34]:
val joinedDf = removedDf.join(df_list) { 관측정점코드 match right.sta_code }

In [37]:
joinedDf
    .select{  일시 and 수온 and sta_name   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 800
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온"}
        line{
            color(sta_name){
             //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ZYJQ3l" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ZYJQ3l");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"sta_name":["영산목포","인천송도","천수만","울산매암","광양적량","시화반월","낙동명지","시화조력","광양초남","광양망덕","새만금","금강하구","영산영암","마산봉암","영산영암","천수만","인천송도","영산목포","시화반월","광양망덕","새만금","금강하구","시화조력","광양적량","울산매암","광양초남","낙동명지","마산봉암","금강하구","시화반월","영산목포","광양망덕","광양적량","천수만","마산봉암","영산영암","광양초남","시화조력","인천송도","울산매암","새만금","낙동명지","광양적량","마산봉암","시화반월","광양망덕","시화조력","영산영암","인천송도","금강하구","울산매암","새만금","낙동명지","광양초남","영산목포","천수만","영산목포","울산매암","금강하구","광양초남","광양적량","시화조력","마산봉암","광양망덕","낙동명지","천수만","새만금","영산영암","시화반월","인천송도","새만금","영산영암","마산봉암","시화반월","광양초남","광양적량","광양망덕","시화조력","인천송도","영산목포","낙동명지","금강하구","울산매암","천수만","시화반월","광양적량","낙동명지","광양망덕","새만금","영산영암","마산봉암","금강하구","시화조력","인천송도","영산목포","울산매암","광양초남","천수만","시화반월","광양적량","천수만","마산봉암","광양초남","낙동명지","시화조력","인천송도","영산목포","울산매암","영산영암","금강하구","새만금","광양망덕","시화조력","광양초남","영산목포","광양적량","인천송도","울산매암","금강하구","마산봉암","낙동명지","시화반월","영산영암","새만금","광양망덕","천수만","광양적량","시화반월","광양망덕","새만금","영산목포","낙동명지","금강하구","울산매암","시화조력","광양초남","마산봉암","영산영암","인천송도","천수만","울산매암","인천송도","시화조력","마산봉암","광양초남","영산영암","시화반월","새만금","광양망덕","광양적량","천수만","금강하구","낙동명지","영산목포","광양초남","금강하구","울산매암","시화조력","마산봉암","시화반월","낙동명지","인천송도","새만금","영산영암","광양망덕","영산목포","광양적량","천수만","광양적량","광양망덕","영산영암","인천송도","천수만","시화반월","울산매암","마산봉암","금강하구","시화조력","낙동명지","영산목포","새만금","광양초남","시화반월","낙동명지","마산봉암","천수만","인천송도","울산매암","영산영암","새만금","광양망덕","영산목포","시화조력","광양초남","광양적량","금강하구","천수만","낙동명지","광양적량","시화반월","울산매암","마산봉암","인천송도","광양초남","광양망덕","금강하구","영산영암","영산목포","새만금","시화조력","광양초남","천수만","시화반월","낙동명지","시화조력","금강하구","광양적량","새만금","영산목포","인천송도","마산봉암","영산영암","울산매암","광양망덕","마산봉암","울산매암","시화조력","새만금","영산영암","광양초남","천수만","시화반월","금강하구","낙동명지","영산목포","광양망덕","인천송도","광양적량","광양적량","시화반월","영산목포","새만금","광양망덕","인천송도","낙동명지","울산매암","영산영암","광양초남","금강하구","마산봉암","시화조력","천수만","마산봉암","금강하구","새만금","영산영암","낙동명지","광양망덕","울산매암","영산목포","천수만","광양적량","시화반월","시화조력","인천송도","광양초남","인천송도","시화반월","시화조력","마산봉암","새만금","광양초남","영산목포","울산매암","광양망덕","낙동명지","천수만","영산영암","금강하구","광양적량","영산영암","새만금","낙동명지","시화조력","금강하구","광양초남","울산매암","인천송도","천수만","광양적량","마산봉암","시화반월","영산목포","광양망덕","금강하구","영산영암","영산목포","천수만","새만금","낙동명지","광양적량","시화조력","시화반월","마산봉암","인천송도","광양초남","울산매암","광양망덕","시화반월","낙동명지","인천송도","금강하구","새만금","광양초남","울산매암","광양망덕","천수만","마산봉암","시화조력","영산목포","영산영암","광양적량","영산영암","시화반월","새만금","광양망덕","울산매암","영산목포","시화조력","인천송도","마산봉암","광양초남","광양적량","천수만","낙동명지","금강하구","낙동명지","시화조력","영산영암","영산목포","광양망덕","광양초남","새만금","시화반월","천수만","인천송도","금강하구","울산매암","마산봉암","광양적량","마산봉암","인천송도","광양초남","광양적량","시화조력","광양망덕","영산목포","낙동명지","시화반월","울산매암","금강하구","새만금","영산영암","천수만","천수만","새만금","광양적량","낙동명지","인천송도","금강하구","광양망덕","영산영암","마산봉암","울산매암","영산목포","시화반월","시화조력","광양초남","울산매암","시화반월","인천송도","마산봉암","금강하구","광양적량","광양초남","새만금","천수만","광양망덕","시화조력","영산목포","영산영암","낙동명지","인천송도","낙동명지","울산매암","영산영암","새만금","영산목포","광양적량","금강하구","광양초남","시화조력","광양망덕","시화반월","마산봉암","천수만","울산매암","광양망덕","영산영암","영산목포","마산봉암","시화반월","금강하구","광양초남","시화조력","광양적량","천수만","인천송도","낙동명지","새만금","울산매암","새만금","광양초남","영산목포","인천송도","낙동명지","시화조력","마산봉암","광양적량","광양망덕","시화반월","영산영암","천수만","금강하구","광양초남","시화반월","금강하구","새만금